# Flight Maneuver Classification - Exploratory Data Analysis

Understand the structure and distribution of flight maneuver accelerometer data.

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Load Data

In [ ]:
# Load training data
train_df = pd.read_csv('../data/train_set.csv')
test_df = pd.read_csv('../data/test_set.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"\nTrain columns: {train_df.columns.tolist()}")
print(f"Test columns: {test_df.columns.tolist()}")

In [ ]:
# Inspect first rows
print("Training data sample:")
print(train_df.head())
print(f"\nLabel distribution:\n{train_df['label'].value_counts().sort_index()}")

## 3. Data Overview

In [ ]:
# Data info
print("Missing values:")
print(train_df.isnull().sum())
print(f"\nUnique maneuvers: {train_df['maneuver_Id'].nunique()}")
print(f"\nMeasurements per maneuver:")
print(train_df.groupby('maneuver_Id').size().describe())

In [ ]:
# Data types and basic statistics
print("Data types:")
print(train_df.dtypes)
print(f"\nBasic statistics:")
print(train_df[['measurement_x', 'measurement_y', 'measurement_z']].describe())

## 4. Label Distribution

In [ ]:
# Visualization: Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train_df['label'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Label Distribution (Raw Counts)')
axes[0].set_ylabel('Count')
axes[0].set_xlabel('Label')

(train_df['label'].value_counts(normalize=True).sort_index() * 100).plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Label Distribution (Percentage)')
axes[1].set_ylabel('Percentage')
axes[1].set_xlabel('Label')
plt.tight_layout()
plt.show()

## 5. Measurement Distributions

In [ ]:
# Visualization: Measurement distributions by label
fig, axes = plt.subplots(3, 1, figsize=(12, 9))
measurements = ['measurement_x', 'measurement_y', 'measurement_z']
for idx, m in enumerate(measurements):
    for label in sorted(train_df['label'].unique()):
        data = train_df[train_df['label'] == label][m]
        axes[idx].hist(data, alpha=0.6, bins=50, label=f'Class {label}')
    axes[idx].set_title(f'Distribution of {m}')
    axes[idx].set_xlabel(m)
    axes[idx].set_ylabel('Frequency')
    axes[idx].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots by label
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for idx, m in enumerate(measurements):
    train_df.boxplot(column=m, by='label', ax=axes[idx])
    axes[idx].set_title(f'{m} by Label')
    axes[idx].set_xlabel('Label')
plt.suptitle('')
plt.tight_layout()
plt.show()

## 6. Correlation Analysis

In [ ]:
# Correlation between measurements
corr = train_df[['measurement_x', 'measurement_y', 'measurement_z']].corr()
print("Correlation matrix:")
print(corr)

# Visualize correlation
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0, 
            square=True, linewidths=1)
plt.title('Correlation between Measurements')
plt.tight_layout()
plt.show()

## 7. Time Series Patterns

In [ ]:
# Sample a few maneuvers to visualize time series patterns
fig, axes = plt.subplots(3, 1, figsize=(14, 9))

for label_val, ax in zip(sorted(train_df['label'].unique()), axes):
    sample_maneuver = train_df[train_df['label'] == label_val]['maneuver_Id'].iloc[0]
    sample_data = train_df[train_df['maneuver_Id'] == sample_maneuver]
    
    ax.plot(sample_data.index, sample_data['measurement_x'], label='x', alpha=0.7)
    ax.plot(sample_data.index, sample_data['measurement_y'], label='y', alpha=0.7)
    ax.plot(sample_data.index, sample_data['measurement_z'], label='z', alpha=0.7)
    
    ax.set_title(f'Sample Maneuver - Class {label_val} (ID: {sample_maneuver})')
    ax.set_xlabel('Time (index)')
    ax.set_ylabel('Measurement Value')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Test Set Overview

In [ ]:
# Test set info
print("Test set sample:")
print(test_df.head())
print(f"\nTest set shape: {test_df.shape}")
print(f"Unique maneuvers: {test_df['maneuver_Id'].nunique()}")
print(f"\nMeasurements per maneuver (test):")
print(test_df.groupby('maneuver_Id').size().describe())

## 9. Summary

In [ ]:
print("\n" + "="*60)
print("DATASET SUMMARY")
print("="*60)
print(f"\nTraining Set:")
print(f"  Total rows: {len(train_df):,}")
print(f"  Unique maneuvers: {train_df['maneuver_Id'].nunique():,}")
print(f"  Class distribution: {train_df['label'].value_counts().sort_index().to_dict()}")
print(f"\nTest Set:")
print(f"  Total rows: {len(test_df):,}")
print(f"  Unique maneuvers: {test_df['maneuver_Id'].nunique():,}")
print(f"\nMeasurements:")
print(f"  Features: measurement_x, measurement_y, measurement_z")
print(f"  Time index: timestamp (MM:SS.S format)")
print(f"\nKey Observations:")
print(f"  - Class imbalance: Class 0 is majority")
print(f"  - Variable maneuver length: {train_df.groupby('maneuver_Id').size().min()} to {train_df.groupby('maneuver_Id').size().max()} measurements")
print(f"  - 3D accelerometer data: x, y, z measurements")
print(f"\nNext Steps:")
print(f"  1. Run flight_maneuver_training.ipynb for model training")
print(f"  2. Run flight_maneuver_submission.ipynb for test predictions")